In [0]:
spark.catalog.setCurrentCatalog("futebol_analytics")

In [0]:
%restart_python

# 00. Provisionamento de Ambiente e Governança (Unity Catalog)
Este notebook automatiza a criação do catálogo customizado `futebol_analytics`, define as camadas lógicas da Arquitetura Medalhão (`bronze`, `silver`, `gold`), gerencia o Volume de arquivos brutos e executa a replicação transacional das tabelas com testes de integridade.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS futebol_analytics;

In [0]:
# 1. Cria o schema e o volume no novo catálogo
spark.sql("CREATE SCHEMA IF NOT EXISTS futebol_analytics.bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS futebol_analytics.bronze.raw_files")

# 2. Copia os CSVs brutos do volume antigo para o novo
caminho_origem = "/Volumes/workspace/bronze/raw_files"
caminho_destino = "/Volumes/futebol_analytics/bronze/raw_files"

for arquivo in dbutils.fs.ls(caminho_origem):
  dbutils.fs.cp(arquivo.path, f"{caminho_destino}/{arquivo.name}")
  print(f"[OK] Arquivo copiado: {arquivo.name}")

# 3. Clona as 16 tabelas Delta de workspace.bronze para futebol_analytics.bronze
tabelas = [t.name for t in spark.catalog.listTables("workspace.bronze")]

for tabela in tabelas:
  spark.sql(f"""
        CREATE OR REPLACE TABLE futebol_analytics.bronze.{tabela} 
        DEEP CLONE workspace.bronze.{tabela}
    """)
  print(f"[OK] Tabela clonada: futebol_analytics.bronze.{tabela}")

In [0]:
tabelas_antigas = spark.sql("SHOW TABLES IN workspace.bronze").count()
tabelas_novas = spark.sql("SHOW TABLES IN futebol_analytics.bronze").count()

print(f"Tabelas em workspace.bronze: {tabelas_antigas}")
print(f"Tabelas em futebol_analytics.bronze: {tabelas_novas}")

# Teste de integridade em uma tabela chave
qtd_origem = spark.table("workspace.bronze.players").count()
qtd_destino = spark.table("futebol_analytics.bronze.players").count()

print(
    f"\nValidação da tabela 'players': {qtd_origem} linhas (origem) vs"
    f" {qtd_destino} linhas (destino)"
)

In [0]:
# Inspecionar as colunas da FotMob
print("--- ESQUEMA FOTMOB ---")
spark.table("futebol_analytics.bronze.fotmob_rating_2022").printSchema()

# Inspecionar as principais tabelas do Kaggle que vamos cruzar
print("\n--- ESQUEMA PLAYERS ---")
spark.table("futebol_analytics.bronze.players").printSchema()

print("\n--- ESQUEMA APPEARANCES ---")
spark.table("futebol_analytics.bronze.appearances").printSchema()